# Figure 4: OBSALL (left) + AQUA (right)

This notebook generates the OBSALL and AQUA panels and combines them into a single publication-ready PDF.

In [1]:
from pathlib import Path
from string import ascii_lowercase
import os

import matplotlib as mpl
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import gridspec
import numpy as np
import pandas as pd
import xarray as xr
from cartopy import crs as ccrs, feature as cfeature

from aqua.core.graphics import plot_single_map_diff, ConfigStyle
from aqua.core.logger import log_configure

/opt/homebrew/Caskroom/miniconda/base/envs/aqua-dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Base paths
root = Path('/Users/mnurisso/src/DestinE_paper/Figure4')
obsall_dir = root / 'OBSALL'
aqua_dir = root / 'AQUA'
output_dir = root / 'figures' / 'combined'
output_dir.mkdir(parents=True, exist_ok=True)

# OBSALL inputs
station_list_path = obsall_dir / 'SYNOP' / 'synop_station_list.txt'
model_results_path = obsall_dir / 'mean-map-model_results.pickle'

# AQUA inputs
netcdf_dir = aqua_dir / 'netcdf'
icon_file_mask = str(netcdf_dir / '*ICON*.nc')
ifs_nemo_file_mask = str(netcdf_dir / '*IFS-NEMO*.nc')
ifs_fesom_file_mask = str(netcdf_dir / '*IFS-FESOM*.nc')
era5_file_mask = str(netcdf_dir / '*ERA5*.nc')

# Output files
obsall_png = output_dir / 'fig4_obsall.png'
aqua_png = output_dir / 'fig4_aqua.png'
combined_pdf = output_dir / 'fig4_obsall_aqua_combined.pdf'

In [44]:
def plot_obsall_panel(output_path: Path, diff_range: float = 7.0, diff_levels: int = 16):
    import pickle

    station_list = (
        pd.read_csv(station_list_path, sep=r"\s+")
        .rename(
            {
                "station@hdr_integer": "id",
                "longitude@hdr:real": "longitude",
                "latitude@hdr:real": "latitude",
                "elevation@hdr:real": "elevation",
            },
            axis="columns",
        )[["id", "longitude", "latitude", "elevation"]]
        .set_index("id")
    )

    with open(model_results_path, "rb") as f:
        model_results = pickle.load(f)

    diff_min = 0.0
    diff_max = 0.0
    for station_stats in model_results.values():
        diff_min = min(
            diff_min,
            (station_stats["ann_mean_sim"] - station_stats["ann_mean_obs"]).min(),
        )
        diff_max = max(
            diff_max,
            (station_stats["ann_mean_sim"] - station_stats["ann_mean_obs"]).max(),
        )

    # diff_extend = {
    #     (True, True): "both",
    #     (True, False): "min",
    #     (False, True): "max",
    #     (False, False): "neither",
    # }[(diff_min < -diff_range, diff_max > diff_range)]
    diff_extend = 'both'

    models = {
        "icon_hist_o25-1": "ICON",
        "ifs-nemo_hist_o25-1": "IFS-NEMO",
        "ifs-fesom_hist_o25-1": "IFS-FESOM",
    }
    combined_models = {
        "icon_hist_o25-1": ["icon_hist_o25-1"],
        "ifs-nemo_hist_o25-1": ["ifs-nemo_hist_o25-1"],
        "ifs-fesom_hist_o25-1": ["ifs-fesom_hist_o25-1"],
    }

    def plot_station_map(ax, station_stats: list[pd.DataFrame]):
        se_factor = 2 * np.sqrt(1 + 1 / len(station_stats))

        ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
        ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.5)
        ax.set_global()

        levels = (
            [-diff_range]
            + list(
                np.linspace(
                    -diff_range / 2, diff_range / 2, diff_levels + (1 - diff_levels % 2) - 2
                )
            )
            + [diff_range]
        )
        cmap = plt.get_cmap("RdBu_r", len(levels) + 1)
        norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend=diff_extend)

        mean_ann_mean_sim = sum(s["ann_mean_sim"] for s in station_stats) / len(station_stats)

        sig_mask = (
            mean_ann_mean_sim
            < (station_stats[0]["ann_mean_obs"] - station_stats[0]["ann_se_obs"] * se_factor)
        ) | (
            mean_ann_mean_sim
            > (station_stats[0]["ann_mean_obs"] + station_stats[0]["ann_se_obs"] * se_factor)
        )

        ax.scatter(
            station_list.loc[mean_ann_mean_sim.index][~sig_mask]["longitude"],
            station_list.loc[mean_ann_mean_sim.index][~sig_mask]["latitude"],
            c=mean_ann_mean_sim.loc[~sig_mask] - station_stats[0].loc[~sig_mask]["ann_mean_obs"],
            cmap=cmap,
            norm=norm,
            s=30,
            marker="^",
            transform=ccrs.PlateCarree(),
            edgecolors="grey",
            lw=0.5,
        )
        sc = ax.scatter(
            station_list.loc[mean_ann_mean_sim.index][sig_mask]["longitude"],
            station_list.loc[mean_ann_mean_sim.index][sig_mask]["latitude"],
            c=mean_ann_mean_sim.loc[sig_mask] - station_stats[0].loc[sig_mask]["ann_mean_obs"],
            cmap=cmap,
            norm=norm,
            s=30,
            marker="o",
            transform=ccrs.PlateCarree(),
            edgecolors="grey",
            lw=0.5,
        )

        legend_elements = [
            mpl.lines.Line2D([0], [0], marker="o", color="black", linestyle="", markersize=5, label="significant"),
            mpl.lines.Line2D([0], [0], marker="^", color="black", linestyle="", markersize=5, label="not significant"),
        ]
        ax.legend(handles=legend_elements, loc="lower left")

        return sc

    fig = plt.figure(figsize=(8, 12))
    gs = gridspec.GridSpec(4, 1, height_ratios=[1, 1, 1, 0.12], hspace=0.25)

    for i, (model, model_name) in enumerate(models.items()):
        a = ascii_lowercase[i]
        ax1 = fig.add_subplot(gs[i], projection=ccrs.Robinson())

        mean_ann_bias = np.mean(
            [
                np.mean(model_results[m]["ann_mean_sim"] - model_results[m]["ann_mean_obs"])
                for m in combined_models[model]
            ]
        )
        mean_ann_abs_bias = np.mean(
            [
                np.mean(np.abs(model_results[m]["ann_mean_sim"] - model_results[m]["ann_mean_obs"]))
                for m in combined_models[model]
            ]
        )
        ann_bias_positive = np.mean(
            [
                np.mean((model_results[m]["ann_mean_sim"] - model_results[m]["ann_mean_obs"]) > 0)
                for m in combined_models[model]
            ]
        )

        ax1.set_title(
            f"{a}) {model_name} ({np.round(mean_ann_bias, 2)}, {np.round(mean_ann_abs_bias, 2)}, {int(np.round(ann_bias_positive * 100))}%)",
            loc="center",
        )
        sc = plot_station_map(ax1, [model_results[m] for m in combined_models[model]])

    cax = fig.add_subplot(gs[3])
    levels = [-7.0, -3.5, -3.0, -2.5, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 7.0]
    cmap = plt.get_cmap("RdBu_r", len(levels) + 1)
    norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend=diff_extend)

    cbar = fig.colorbar(
        sc,
        cax=cax,
        cmap=cmap,
        norm=norm,
        extend=diff_extend,
        boundaries=levels,
        ticks=levels,
        orientation="horizontal",
        label="2m Temperature Bias ($^{o}C$)",
    )
    for boundary in levels:
        cbar.ax.vlines(boundary, *cbar.ax.get_ylim(), color="black", linewidth=0.7)

    pos = cax.get_position()
    #cax.set_position([0.24, pos.y0, 0.51, pos.height])
    cax.set_position([0.22, pos.y0, 0.57, pos.height])

    plt.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.close(fig)


def plot_aqua_panel(output_path: Path):
    loglevel = "WARNING"
    logger = log_configure(loglevel, "plot_biases")
    ConfigStyle(loglevel=loglevel)

    icon_climatology = xr.open_mfdataset(icon_file_mask)
    ifs_nemo_climatology = xr.open_mfdataset(ifs_nemo_file_mask)
    ifs_fesom_climatology = xr.open_mfdataset(ifs_fesom_file_mask)
    era5_climatology = xr.open_mfdataset(era5_file_mask)

    models = [icon_climatology, ifs_nemo_climatology, ifs_fesom_climatology]
    titles = ["ICON", "IFS-NEMO", "IFS-FESOM"]
    panels_label = ["d)", "e)", "f)"]
    for i in range(len(models)):
        titles[i] = f"{panels_label[i]} {titles[i]}"
    var = "u"

    nrows = len(models)
    ncols = 1
    fig = plt.figure(figsize=(8, 12))
    levels = np.arange(-7.5, 7.5 + 1, 1)
    cmap = "PuOr_r"
    norm = None

    for i, model in enumerate(models, 1):
        data = model[var]
        fig, ax = plot_single_map_diff(
            data=data,
            data_ref=era5_climatology[var],
            fig=fig,
            norm=norm,
            ax_pos=(nrows, ncols, i),
            cmap=cmap,
            title=titles[i - 1],
            vmin_fill=-7.5,
            vmax_fill=7.5,
            vmin_contour=-10.0,
            vmax_contour=15.0,
            return_fig=True,
            cbar=False,
            sym=False,
            line_levels=6,
            nlevels=15,
            title_size=14,
            loglevel=loglevel,
        )
        # ax.text(
        #     0.02,
        #     0.95,
        #     panels_label[i - 1],
        #     transform=ax.transAxes,
        #     fontsize=12,
        #     fontweight="bold",
        #     va="top",
        #     ha="left",
        # )

    fig.subplots_adjust(bottom=0.20, top=0.87, left=0.05, right=0.95, wspace=0.1, hspace=0.2)
    cbar_ax = fig.add_axes([0.2, 0.15, 0.6, 0.03])
    mappable = ax.collections[0]
    cbar = fig.colorbar(mappable, cax=cbar_ax, orientation="horizontal", label="Zonal Wind Speed (m/s)")
    cbar.set_ticks(levels)
    cbar.ax.set_xticklabels([f"{tick:.0f}" if tick == int(tick) else f"{tick:.1f}" for tick in levels])
    cbar.ax.tick_params(labelsize=9)

    for boundary in levels:
        cbar.ax.vlines(boundary, *cbar.ax.get_ylim(), color="black", linewidth=0.7)

    fig.savefig(output_path, dpi=300)
    plt.close(fig)
    logger.info(f"Figure saved to {output_path}")

In [46]:
plot_obsall_panel(obsall_png)
plot_aqua_panel(aqua_png)

/var/folders/8s/zx3xtpwn4xq3sq3ln4cp3c300000gn/T/ipykernel_98701/1734611461.py:169: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [ ]:
obs_img = mpimg.imread(obsall_png)
aqua_img = mpimg.imread(aqua_png)

fig = plt.figure(figsize=(12, 10))
gs = gridspec.GridSpec(1, 2, wspace=0.0)

ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(obs_img, interpolation="none", aspect="auto")
ax1.set_anchor("N")
ax1.axis("off")

ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(aqua_img, interpolation="none", aspect="auto")
ax2.set_anchor("N")
ax2.axis("off")

fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01, wspace=0.0)
fig.savefig(combined_pdf, dpi=300, bbox_inches="tight", pad_inches=0.02)
plt.close(fig)
combined_pdf

UnidentifiedImageError: cannot identify image file '/Users/mnurisso/src/DestinE_paper/Figure4/figures/combined/fig4_obsall.pdf'